# AI Lab 5

## Task 1

Enhanced Maze Navigation with Multiple Goals
● Description: Modify the given Best-First Search to find a path through a maze
with multiple goal points. The algorithm should visit all goal points and return
the shortest path covering all goals.
● Challenge: The maze will have several dead ends and multiple goal points at
different locations.

In [1]:
from collections import deque 

def shortest_path_all_goals(maze):
    rows, cols = len(maze), len(maze[0])

    start = None
    goals = []
    for r in range(rows):
        for c in range(cols):
            if maze[r][c] == "S":
                start = (r, c)
            elif maze[r][c] == "G":
                goals.append((r, c))

    if start is None or not goals:
        return None

    goal_id = {g: i for i, g in enumerate(goals)}
    all_mask = (1 << len(goals)) - 1

    sr, sc = start
    start_mask = 0
    if start in goal_id:
        start_mask |= 1 << goal_id[start]

    q = deque([(sr, sc, start_mask)])
    parent = {(sr, sc, start_mask): None}

    dirs = [(1, 0), (-1, 0), (0, 1), (0, -1)]

    while q:
        r, c, mask = q.popleft()

        if mask == all_mask:
            path = []
            state = (r, c, mask)
            while state is not None:
                path.append((state[0], state[1]))
                state = parent[state]
            return path[::-1]

        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < rows and 0 <= nc < cols):
                continue
            if maze[nr][nc] == "#":
                continue

            nmask = mask
            if (nr, nc) in goal_id:
                nmask |= 1 << goal_id[(nr, nc)]

            nxt = (nr, nc, nmask)
            if nxt not in parent:
                parent[nxt] = (r, c, mask)
                q.append(nxt)

    return None


def draw_path(maze, path):
    grid = [list(row) for row in maze]
    for r, c in path:
        if grid[r][c] == ".":
            grid[r][c] = "*"
    return ["".join(row) for row in grid]


if __name__ == "__main__":
    maze = [
        "###############",
        "#S....#...#...#",
        "###.#.#.#.#.#G#",
        "#...#...#...#.#",
        "#.#####.#####.#",
        "#...G...#.....#",
        "#.#.###.#.###.#",
        "#.#.....#...G.#",
        "###############",
    ]

    path = shortest_path_all_goals(maze)

    if path is None:
        print("No path found that visits all goals.")
    else:
        print("Shortest path length:", len(path) - 1)
        print("Path:", path)
        for row in draw_path(maze, path):
            print(row)

Shortest path length: 35
Path: [(1, 1), (1, 2), (1, 3), (2, 3), (3, 3), (3, 2), (3, 1), (4, 1), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (5, 6), (5, 7), (4, 7), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (2, 9), (3, 9), (3, 10), (3, 11), (2, 11), (1, 11), (1, 12), (1, 13), (2, 13), (3, 13), (4, 13), (5, 13), (6, 13), (7, 13), (7, 12)]
###############
#S**..#***#***#
###*#.#*#*#*#G#
#***#..*#***#*#
#*#####*#####*#
#***G***#....*#
#.#.###.#.###*#
#.#.....#...G*#
###############


## Task 2

Implement an A* Search where the edge costs change dynamically at random intervals.
The algorithm should adapt to these changes and always find the optimal path.
Recompute and adjust paths in real time without restarting the algorithm from
scratch.

In [2]:
import heapq
import math
import random


INF = math.inf
DIRECTIONS = ((1, 0), (-1, 0), (0, 1), (0, -1))


def build_world(grid):
    rows = len(grid)
    cols = len(grid[0])
    blocked = set()
    cost = {}
    for r in range(rows):
        for c in range(cols):
            node = (r, c)
            if grid[r][c] == "#":
                blocked.add(node)
                cost[node] = INF
            else:
                cost[node] = 1.0
    return rows, cols, blocked, cost


def in_bounds(node, rows, cols):
    r, c = node
    return 0 <= r < rows and 0 <= c < cols


def neighbors(node, rows, cols, blocked):
    r, c = node
    for dr, dc in DIRECTIONS:
        nxt = (r + dr, c + dc)
        if in_bounds(nxt, rows, cols) and nxt not in blocked:
            yield nxt


def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def a_star(rows, cols, blocked, cost, start, goal):
    open_heap = [(0.0, start)]
    parent = {start: None}
    g = {start: 0.0}

    while open_heap:
        _, current = heapq.heappop(open_heap)
        if current == goal:
            break

        for nxt in neighbors(current, rows, cols, blocked):
            new_cost = g[current] + cost[nxt]
            if new_cost < g.get(nxt, INF):
                g[nxt] = new_cost
                parent[nxt] = current
                f = new_cost + heuristic(nxt, goal)
                heapq.heappush(open_heap, (f, nxt))

    if goal not in parent:
        return None, INF

    path = []
    cur = goal
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    path.reverse()
    return path, g[goal]


def render(rows, cols, blocked, start, goal, path=None):
    board = []
    path_set = set(path or [])
    for r in range(rows):
        row = []
        for c in range(cols):
            node = (r, c)
            if node == start:
                row.append("S")
            elif node == goal:
                row.append("G")
            elif node in blocked:
                row.append("#")
            elif node in path_set:
                row.append("*")
            else:
                row.append(".")
        board.append("".join(row))
    return "\n".join(board)


def apply_random_updates(blocked, cost, start, goal, mutable_nodes, updates_per_tick):
    updates_per_tick = min(updates_per_tick, len(mutable_nodes))
    changes = []
    for node in random.sample(mutable_nodes, updates_per_tick):
        new_cost = INF if random.random() < 0.10 else float(random.randint(1, 6))
        if node in (start, goal):
            continue
        if math.isinf(new_cost):
            blocked.add(node)
            cost[node] = INF
        else:
            blocked.discard(node)
            cost[node] = new_cost
        changes.append((node, new_cost))
    return changes


def main():
    random.seed(42)

    grid = [
        "S...#.....",
        ".#..#..#..",
        "....#..#..",
        "##..#.....",
        "....###...",
        "..#....#..",
        "..#..#....",
        "..#..#..G.",
    ]

    start = (0, 0)
    goal = (7, 8)
    rows, cols, blocked, cost = build_world(grid)

    mutable_nodes = [
        (r, c) for r in range(rows) for c in range(cols) if (r, c) not in {start, goal}
    ]

    print("Legend: S=start, G=goal, #=blocked, *=current best path")
    print("Tick 0 is the baseline map before random cost changes.\n")

    path, total_cost = a_star(rows, cols, blocked, cost, start, goal)
    print("=== Tick 0 (baseline) ===")
    if path is None:
        print("No valid path in baseline map.")
    else:
        print(f"Optimal path cost: {total_cost:.0f}")
        print(render(rows, cols, blocked, start, goal, path))

    for tick in range(1, 9):
        changes = apply_random_updates(blocked, cost, start, goal, mutable_nodes, updates_per_tick=4)
        path, total_cost = a_star(rows, cols, blocked, cost, start, goal)

        print(f"\n=== Tick {tick} ===")
        print("Updates:")
        for node, new_cost in changes:
            label = "BLOCKED" if math.isinf(new_cost) else int(new_cost)
            print(f"  {node} -> {label}")

        if path is None:
            print("No valid path right now.")
        else:
            print(f"Optimal path cost: {total_cost:.0f}")
            print(render(rows, cols, blocked, start, goal, path))


if __name__ == "__main__":
    main()

Legend: S=start, G=goal, #=blocked, *=current best path
Tick 0 is the baseline map before random cost changes.

=== Tick 0 (baseline) ===
Optimal path cost: 15
S***#.....
.#.*#..#..
...*#..#..
##.*#.....
...*###...
..#****#..
..#..#***.
..#..#..G.

=== Tick 1 ===
Updates:
  (1, 5) -> 6
  (0, 4) -> 6
  (3, 6) -> 1
  (3, 2) -> 1
Optimal path cost: 15
S***......
.#.*#..#..
...*#..#..
##.*#.....
...*###...
..#****#..
..#..#***.
..#..#..G.

=== Tick 2 ===
Updates:
  (0, 4) -> 1
  (1, 2) -> 6
  (2, 8) -> 5
  (3, 0) -> 4
Optimal path cost: 15
S******...
.#..#.*#..
....#.*#..
.#..#.***.
....###.*.
..#....#*.
..#..#..*.
..#..#..G.

=== Tick 3 ===
Updates:
  (7, 6) -> 3
  (3, 6) -> 2
  (0, 1) -> 3
  (2, 1) -> 4
Optimal path cost: 17
S***......
.#.*#..#..
...*#..#..
.#.*#.....
...*###...
..#****#..
..#..#***.
..#..#..G.

=== Tick 4 ===
Updates:
  (1, 3) -> 1
  (4, 6) -> 5
  (4, 5) -> 4
  (7, 9) -> BLOCKED
Optimal path cost: 17
S***......
.#.*#..#..
...*#..#..
.#.*#.....
...*#.....
..#****#..
..#.

## Task 3

Delivery Route Optimization with Time Windows
● Description: Using the Greedy Best-First Search, optimize delivery routes for a
set of delivery points. Each delivery point has a specific time window for
delivery, and the algorithm must prioritize those with stricter deadlines.
● Challenge: Ensure that the algorithm handles time constraints efficiently while
minimizing total travel distance.

In [3]:
import heapq
import math


def distance(a, b):
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)


def greedy_best_first_delivery(depot, deliveries):
    unvisited = list(range(len(deliveries)))
    current_pos = depot
    current_time = 0.0
    route = [("Depot", depot)]

    while unvisited:
        best = None
        best_score = float("inf")

        for i in unvisited:
            name, pos, deadline = deliveries[i]
            dist = distance(current_pos, pos)
            arrival = current_time + dist
            urgency = deadline - arrival
            score = urgency
            if score < best_score:
                best_score = score
                best = i

        name, pos, deadline = deliveries[best]
        travel = distance(current_pos, pos)
        current_time += travel
        current_pos = pos
        status = "On Time" if current_time <= deadline else "Late"
        route.append((name, pos, round(current_time, 2), deadline, status))
        unvisited.remove(best)

    travel_back = distance(current_pos, depot)
    current_time += travel_back
    route.append(("Back to Depot", depot, round(current_time, 2)))
    return route


def main():
    depot = (0, 0)

    deliveries = [
        ("A", (2, 3), 10),
        ("B", (5, 1), 8),
        ("C", (8, 7), 20),
        ("D", (1, 8), 12),
        ("E", (6, 4), 15),
    ]

    route = greedy_best_first_delivery(depot, deliveries)

    print("Delivery Route (Greedy Best-First by urgency):\n")
    for stop in route:
        if len(stop) == 2:
            print(f"  Start: {stop[0]} at {stop[1]}")
        elif len(stop) == 3:
            print(f"  {stop[0]} at {stop[1]}  |  Time: {stop[2]}")
        else:
            name, pos, arrival, deadline, status = stop
            print(f"  {name} at {pos}  |  Arrival: {arrival}  Deadline: {deadline}  [{status}]")


if __name__ == "__main__":
    main()

Delivery Route (Greedy Best-First by urgency):

  Start: Depot at (0, 0)
  B at (5, 1)  |  Arrival: 5.1  Deadline: 8  [On Time]
  D at (1, 8)  |  Arrival: 13.16  Deadline: 12  [Late]
  A at (2, 3)  |  Arrival: 18.26  Deadline: 10  [Late]
  E at (6, 4)  |  Arrival: 22.38  Deadline: 15  [Late]
  C at (8, 7)  |  Arrival: 25.99  Deadline: 20  [Late]
  Back to Depot at (0, 0)  |  Time: 36.62
